In [1]:
!pip install scikit-learn pandas

In [5]:

import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split


# Load Dataset


print("Loading Dataset...")

df = pd.read_csv("IMDB Dataset.csv")

# Use only first 5000 samples

df = df.head(5000)

# Converting labels to numbers

df["sentiment"] = df["sentiment"].map({
    "positive": 1,
    "negative": 0
})

print(df.head())

# Convert Text to Numerical Form

vectorizer = TfidfVectorizer(max_features=3000)

X = vectorizer.fit_transform(df["review"])

y = df["sentiment"]

# Train-Test Split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Simulate 3 Federated Clients

client_size = X_train.shape[0] // 3

X_client1 = X_train[:client_size]
y_client1 = y_train[:client_size]

X_client2 = X_train[client_size:2*client_size]
y_client2 = y_train[client_size:2*client_size]

X_client3 = X_train[2*client_size:]
y_client3 = y_train[2*client_size:]

# Local Training Function

def train_local_model(X_local, y_local):

    model = LogisticRegression(max_iter=1000)

    model.fit(X_local, y_local)

    return model

# Train Client Models

print("Training Client 1...")
model1 = train_local_model(X_client1, y_client1)

print("Training Client 2...")
model2 = train_local_model(X_client2, y_client2)

print("Training Client 3...")
model3 = train_local_model(X_client3, y_client3)

# Federated Averaging (FedAvg)

avg_coef = (
    model1.coef_
    + model2.coef_
    + model3.coef_
) / 3

avg_intercept = (
    model1.intercept_
    + model2.intercept_
    + model3.intercept_
) / 3

# Create Global Model

global_model = LogisticRegression(max_iter=1000)

# Train once to initialize

global_model.fit(X_train, y_train)

# Replace parameters with FedAvg result

global_model.coef_ = avg_coef
global_model.intercept_ = avg_intercept


# Evaluate Model

predictions = global_model.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

print("\nFederated Learning Completed")

print("Global Model Accuracy:", accuracy)


# Test Custom Review

while True:

    user_input = input("\nEnter a movie review (or type exit): ")

    if user_input.lower() == "exit":
        break

    transformed = vectorizer.transform([user_input])

    prediction = global_model.predict(transformed)

    if prediction[0] == 1:
        print("Prediction: Positive")
    else:
        print("Prediction: Negative")

Loading Dataset...
                                              review  sentiment
0  One of the other reviewers has mentioned that ...          1
1  A wonderful little production. <br /><br />The...          1
2  I thought this was a wonderful way to spend ti...          1
3  Basically there's a family where a little boy ...          0
4  Petter Mattei's "Love in the Time of Money" is...          1
Training Client 1...
Training Client 2...
Training Client 3...

Federated Learning Completed
Global Model Accuracy: 0.851

Enter a movie review (or type exit): Beautiful
Prediction: Positive

Enter a movie review (or type exit): Worst
Prediction: Negative

Enter a movie review (or type exit): Moderate
Prediction: Positive

Enter a movie review (or type exit): Sava comsi comsa
Prediction: Positive

Enter a movie review (or type exit): Sava maal
Prediction: Positive

Enter a movie review (or type exit): Sava mal
Prediction: Positive

Enter a movie review (or type exit): Could be more better
P